# ConversationKGMemory

지식 그래프의 힘을 활용하여 정보를 저장하고 불러옵니다.

이를 통해 모델이 서로 다른 개체 간의 관계를 이해하는 데 도움을 주고, 복잡한 연결망과 역사적 맥락을 기반으로 대응하는 능력을 향상시킵니다.


In [7]:
!pip install python-dotenv
!pip install -U langchain
!pip install langchain -U langchain_openai
!pip install langchain_core
!pip install langchain_classic
!pip install langchain_community
!pip install -U networkx

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 4.8 MB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.1 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 4.3 MB/s  0:00:00


In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationKGMemory

In [14]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

memory = ConversationKGMemory(llm=llm, return_messages=True)
memory.save_context(
    {"input": "이쪽은 세종에 거주중인 김희영입니다."},
    {"output": "김희영씨는 누구시죠?"},
)
memory.save_context(
    {"input": "김희영씨는 우리 회사의 PMO입니다."},
    {"output": "만나서 반갑습니다. 잘 부탁 드립니다."}
)

In [15]:
memory.load_memory_variables({"input": "김희영씨는 누구입니까?"})

{'history': [SystemMessage(content='On 김희영: 김희영 is PMO. 김희영 is in 우리 회사.', additional_kwargs={}, response_metadata={})]}

## Chain 에 메모리 활용하기

`ConversationChain` 에 `ConversationKGMemory` 를 메모리로 지정하여 대화를 나눈 후 memory 를 확인해 보도록 하겠습니다.

In [13]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains import ConversationChain

llm = ChatOpenAI(temperature=0)

template = """The following is a friendly conversation between a human and an AI. 
The AI is talkative and provides lots of specific details from its context. 
If the AI does not know the answer to a question, it truthfully says it does not know. 
The AI ONLY uses information contained in the "Relevant Information" section and does not hallucinate.

Relevant Information:

{history}

Conversation:
Human: {input}
AI:"""

prompt = PromptTemplate(
    input_variables=["history", "input"], template=template)

conversation_with_kg = ConversationChain(
    llm=llm, prompt=prompt, memory=ConversationKGMemory(llm=llm)
)

C:\Users\user\AppData\Local\Temp\ipykernel_17796\3965292462.py:22: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation_with_kg = ConversationChain(


In [19]:
conversation_with_kg.predict(
    input="My name is Heeyeong. she is a coworker of mine who charge of PMO at our company"
)

"Hello Heeyeong! It's nice to meet you. I see that Min is in charge of PMO at your company. How can I assist you today?"

In [18]:
conversation_with_kg.memory.load_memory_variables({"input": "who is Heeyeong?"})

{'history': 'On Heeyeong: Heeyeong is a coworker of Min. Heeyeong is a coworker. Heeyeong works at company.'}